# LongMemEval domain fine-tune (T4 Colab)

Self-contained: clones the experiment branch, downloads LongMemEval-s
from HuggingFace (~265 MB), runs the data prep + ingest, then trains
with `vstash retrain --training-queries ... --eval-queries ...`
(non-synthetic, labeled v5 recipe). No Drive upload needed.

Reported as a SEPARATE result from the LongMemEval baseline
retrieval numbers: this is a domain-adaptation demonstration
("training on chat data improves chat retrieval"), not a way to
inflate v3's headline number.

## GPU + persistence

Training auto-uses CUDA via `SentenceTransformer.fit()` whenever
`torch.cuda.is_available()` is True (cell 4 verifies).

| Step | Where it runs | Device source |
|---|---|---|
| Bulk-mine | GPU | `--bulk-mine-device cuda` (explicit) |
| Train MNRL | GPU | SentenceTransformer auto-detect |
| Eval baseline + final | GPU | SentenceTransformer auto-detect |
| Ingest (cell 3) | CPU | FastEmbed ONNX (no CUDA wheel) |

**Disconnect-safe layout (lesson from prior run):** Drive is
mounted in cell 4 *before* the long step.  Cells 5 and 7 copy
the trained model to Drive **as soon as the retrain CLI returns**
(not in a separate cell), so a runtime disconnect after training
does not lose the model.  If the runtime dies *during* training,
rerun cells 1-4 then jump straight to the retrain cell -- the
ingested corpus DB lives on `/content/` which is wiped on
disconnect, so prep (cell 3) must rerun unless `/content/lme_corpus.db`
was also persisted to Drive (cell 4 covers that).

**Long-run keepalive:** keep the Colab tab focused or use a small
JS snippet to ping the runtime; the free-tier idle timeout kicks
in after ~90 min of no UI activity.  Browser dev-tools console:
`setInterval(() => document.querySelector('#top-toolbar').click(), 60000)`.

In [ ]:
# Cell 1: Setup -- clone the branch with the LongMemEval experiment
# scripts (lme_prepare_retrain, longmemeval_retrieval).  --eval-queries
# is on develop already (PR #299) so any develop-or-later branch works.
BRANCH = 'feature/longmemeval-retrain-experiments'

!pip install -q 'sentence-transformers>=3' torch 'accelerate>=1.1.0' huggingface_hub
!rm -rf /content/vstash
!git clone --branch $BRANCH https://github.com/stffns/vstash.git /content/vstash
%cd /content/vstash
!pip install -q -e .
!vstash --version
!vstash retrain --help | grep -E '\-\-training-queries|\-\-eval-queries' | head -4

In [ ]:
# Cell 2: Download longmemeval_s (~265 MB) from HF directly into the
# experiments/data/longmemeval/ path that the prep script expects.
import os
from huggingface_hub import hf_hub_download

TARGET = '/content/vstash/experiments/data/longmemeval'
os.makedirs(TARGET, exist_ok=True)

path = hf_hub_download(
    repo_id='xiaowu0162/longmemeval',
    filename='longmemeval_s',
    repo_type='dataset',
    local_dir=TARGET,
)
print(f'Downloaded longmemeval_s ({os.path.getsize(path) / 1024 / 1024:.1f} MB) -> {path}')

In [ ]:
# Cell 3: Build the corpus DB + train/eval JSONLs.
# Stratified 80/20 split by question_type, deterministic seed=42.
# ~9 min on Colab CPU before the GPU phase begins (ingest is
# embedding-bound and FastEmbed has no CUDA wheel; the GPU phase
# is bulk_mine + train + eval).
import os
os.chdir('/content/vstash')

!python -m experiments.lme_prepare_retrain \
    --output-db    /content/lme_corpus.db \
    --output-train /content/lme_train.jsonl \
    --output-eval  /content/lme_eval.jsonl \
    --output-meta  /content/lme_retrain_meta.json \
    --force

import json
meta = json.load(open('/content/lme_retrain_meta.json'))
print()
print('Split:')
for k in ('n_train_questions', 'n_holdout_questions', 'n_train_qrels',
         'n_eval_qrels', 'n_train_docs', 'n_holdout_docs'):
    print(f'  {k:25s} {meta[k]}')
print('By type:')
for t, counts in meta['by_type'].items():
    print(f'  {t:30s} train={counts["train"]:3d}  holdout={counts["holdout"]:3d}')

In [ ]:
# Cell 4: Mount Drive UP FRONT (before the long step) and verify
# the entire retrain pipeline will run on T4 GPU.  This avoids
# the prior failure mode where the runtime disconnected after
# training and the post-hoc Drive copy never happened.
import os
from google.colab import drive
drive.mount('/content/drive')
DRIVE_OUT = '/content/drive/MyDrive/lme_retrain'
os.makedirs(DRIVE_OUT, exist_ok=True)
print('Drive mounted, output dir:', DRIVE_OUT)

# Persist the corpus + meta + JSONLs to Drive too, so re-runs after
# a disconnect can skip the 9-min ingest step (cell 3) by copying
# back from Drive instead of regenerating.
for fname in ('lme_corpus.db', 'lme_train.jsonl', 'lme_eval.jsonl', 'lme_retrain_meta.json'):
    src = f'/content/{fname}'
    dst = f'{DRIVE_OUT}/{fname}'
    if os.path.exists(src) and not os.path.exists(dst):
        !cp "$src" "$dst"
        print(f'  saved {fname} -> Drive')

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['VSTASH_DB_PATH'] = '/content/lme_corpus.db'

import torch
from sentence_transformers import SentenceTransformer

assert torch.cuda.is_available(), (
    'CUDA is not available. Runtime -> Change runtime type -> T4 GPU.'
)
print('torch:', torch.__version__)
print('cuda device:', torch.cuda.get_device_name(0))
print('mem GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

_probe = SentenceTransformer('BAAI/bge-small-en-v1.5')
assert _probe.device.type == 'cuda', (
    f'SentenceTransformer landed on {_probe.device}; train will use CPU. '
    'Restart runtime and re-check.'
)
print('SentenceTransformer device:', _probe.device, '-- training will use GPU.')
del _probe; torch.cuda.empty_cache()

BASE_MODEL = 'BAAI/bge-small-en-v1.5'
OUTPUT_PATH = '/content/bge-small-rrf-lme-v1'
MAX_QUERIES = 5000
EPOCHS = 2
LR = 3e-6
BATCH = 64
SEED = 42

In [ ]:
# Cell 5: Run vstash retrain + IMMEDIATELY persist to Drive.
#
# All three GPU steps:
#   - generate_labeled_triples_batched: --bulk-mine-device cuda (explicit).
#   - train MNRL: SentenceTransformer auto-detects CUDA (verified in cell 4).
#   - evaluate_model baseline + final: same auto-detect.
#
# Gate refuses to save the candidate if NDCG@10 does not improve
# (--min-gain 0.0); the .candidate dir is left in place either way
# so we copy both the promoted and candidate paths to Drive.
import os, time
t0 = time.perf_counter()
!vstash retrain \
    --training-queries /content/lme_train.jsonl \
    --eval-queries     /content/lme_eval.jsonl \
    --base-model       $BASE_MODEL \
    --output           $OUTPUT_PATH \
    --max-queries      $MAX_QUERIES \
    --epochs           $EPOCHS \
    --lr               $LR \
    --batch-size       $BATCH \
    --seed             $SEED \
    --bulk-mine        \
    --bulk-mine-device cuda
print(f'\n[retrain base BGE] wall: {time.perf_counter() - t0:.1f}s')

# Persist EVERYTHING training-related the moment the CLI returns,
# before any runtime hiccup can wipe /content/.
for suffix in ('', '.candidate', '.old'):
    src = OUTPUT_PATH + suffix
    if os.path.isdir(src):
        dst = f'{DRIVE_OUT}/bge-small-rrf-lme-v1{suffix}'
        !rm -rf "$dst" && cp -r "$src" "$dst"
        print(f'  Drive <- {os.path.basename(src)}')
    else:
        print(f'  (skip) {src} does not exist')

# Belt-and-suspenders: confirm the saved model loads on CUDA.
import torch
from sentence_transformers import SentenceTransformer
_check_path = OUTPUT_PATH if os.path.isdir(OUTPUT_PATH) else OUTPUT_PATH + '.candidate'
if os.path.isdir(_check_path):
    _check = SentenceTransformer(_check_path)
    print('saved model device:', _check.device, '| dim:', _check.get_sentence_embedding_dimension())
    del _check; torch.cuda.empty_cache()

In [ ]:
# Cell 6 (optional): also push the candidate to HuggingFace as a
# more robust off-machine backup than Drive (Drive can occasionally
# silently hiccup on large dirs).  Skip if you do not want a public
# upload yet -- the model on Drive (cell 5) is already enough for
# the local Mac to pull and benchmark.
#
# Requires `huggingface_hub login` (token with write scope).  Set
# DRY_RUN=False to actually upload.
DRY_RUN = True
HF_REPO = 'Stffens/bge-small-rrf-lme-v1'

import os
candidate_dir = OUTPUT_PATH if os.path.isdir(OUTPUT_PATH) else OUTPUT_PATH + '.candidate'
if not os.path.isdir(candidate_dir):
    print('No candidate to upload.')
elif DRY_RUN:
    print(f'[dry run] would upload {candidate_dir} -> {HF_REPO}')
else:
    from huggingface_hub import HfApi, login
    # login() opens a browser flow; stash a token in a Colab secret if
    # you want non-interactive upload.
    login()
    api = HfApi()
    api.create_repo(HF_REPO, exist_ok=True)
    api.upload_folder(
        folder_path=candidate_dir,
        repo_id=HF_REPO,
        commit_message='LongMemEval domain fine-tune from base BGE',
    )
    print(f'Uploaded -> https://huggingface.co/{HF_REPO}')

In [ ]:
# Cell 7 (optional): try v3 as base model.  Same disconnect-safe
# pattern as cell 5 -- copy to Drive immediately after the CLI
# returns.
#
# The two arms answer different questions:
#   base BGE -> bge-small-rrf-lme-v1: pure 'chat data lifts BGE'.
#   v3       -> bge-small-rrf-lme-v1-from-v3: 'chat data on top of
#               BEIR-tuned weights still lifts further?'.
import os, time
OUTPUT_V3 = '/content/bge-small-rrf-lme-v1-from-v3'
BASE_V3   = 'Stffens/bge-small-rrf-v3'
t0 = time.perf_counter()
!vstash retrain \
    --training-queries /content/lme_train.jsonl \
    --eval-queries     /content/lme_eval.jsonl \
    --base-model       $BASE_V3 \
    --output           $OUTPUT_V3 \
    --max-queries      $MAX_QUERIES \
    --epochs           $EPOCHS \
    --lr               $LR \
    --batch-size       $BATCH \
    --seed             $SEED \
    --bulk-mine        \
    --bulk-mine-device cuda
print(f'\n[retrain v3-base] wall: {time.perf_counter() - t0:.1f}s')

for suffix in ('', '.candidate', '.old'):
    src = OUTPUT_V3 + suffix
    if os.path.isdir(src):
        dst = f'{DRIVE_OUT}/bge-small-rrf-lme-v1-from-v3{suffix}'
        !rm -rf "$dst" && cp -r "$src" "$dst"
        print(f'  Drive <- {os.path.basename(src)}')